In [ ]:
import coffea
import coffea.hist as hist
import uproot
import matplotlib.pyplot as plt
import mplhep as hep
import numpy as np

import awkward as ak

from sklearn.cluster import DBSCAN

from coffea.nanoevents import NanoEventsFactory, NanoAODSchema,BaseSchema

In [ ]:
from collections import OrderedDict

# Load ntuples
Here we will load ntuples for signal and data (high-MET skim), where the clusters are already reconstructed and stored in the ntuples.

We will use these ntuples to plot some of the key variables that will used to veto background clusters

In [ ]:
fpath={}
tree ={}


# load high-MET skim data
path = '/eos/uscms/store/user/cmsdas/2025/long_exercises/MDS/data/'
fpath['data'] = path + 'Run2_displacedJetMuonNtupler_V1p17_Data2016_Data2017_Data2018-HighMET_goodLumi.root'

mass = [40]
OLD_CTAU = np.array([1000, 10000, 100000])#in mm

path = '/eos/uscms/store/user/cmsdas/2025/long_exercises/MDS/signal/nocuts/'

for m in mass:
    for ct in OLD_CTAU:
        key = 'MC_'+str(m)+'_'+str(ct)                       
        fpath[key] = path +  'ggH_HToSSTobbbb_MH-125_MS-40_ctau-'+str(ct)+'_137000pb_weighted.root'
        
        
for k,v in fpath.items():
    tree[k] = NanoEventsFactory.from_root(v,
                                   schemaclass=BaseSchema,
                                   treepath="MuonSystem"
                                  ).events()

In [ ]:
# See what's contained in the tree dictionary
tree

# Trigger selection

Trigger selection is the beginning of any CMS analysis. 

We are using `MET > 200 GeV` due to the lack of dedicated trigger on the MDS object. 

As a result, in order for signal events to pass the large MET trigger
 - the Higgs is recoiled against a high pT jet
 - When the LLPs decays beyond calorimeters, the decay
 - Since the Higgs has a high pT, the MET will be large
 
You will try to confirm this for yourself first.

 - Use the `met` branch to access the reconstructed MET
 - Use the `gHiggsPt` branch to access the generated Higgs pT  

# Ex: Plot the MET and Higgs pT for signals

 - Focus on the region beyond 200 GeV
 - Q: What do you observe?

**Answer (Check your understanding, MET vs Higgs pT):** The MET and generator Higgs pT distributions have a similar shape above 200 GeV. The Higgs is recoiled against the ISR jet, and the LLPs decay beyond the calorimeters, so the MET is the (invisible) transverse momentum of the Higgs system. This is why a high-pT Higgs yields large MET and passes the MET > 200 GeV trigger. Quote the printed numbers from your own run for the exact shape.


In [ ]:
h= hist.Hist("Events",
              hist.Cat("var","Variable"),
              hist.Bin("pT", "MET/Higgs pT[GeV]", 50, 200, 1200)
             )

sample = "MC_40_1000"
# Fill the code
h.fill(var = "MET", pT = tree[sample].met )
h.fill(var = "Higgs pT", pT = tree[sample].gHiggsPt )

ax = hist.plot1d(h,density=True)
ax.legend(title = sample)
ax.set_ylabel("Fraction of events / bin width")
hep.cms.label(ax=ax,label="Preliminary")


# Ex: Trigger efficiency for signals

In [ ]:
plt.style.use(hep.style.CMS)  

In [ ]:
h_MET= hist.Hist("Events",
              hist.Cat("sample","sample"),
              hist.Bin("met", "MET[GeV]", 50, 0, 500)
             )

for m in mass:
    for ct in OLD_CTAU:
        # Fill the histogram h_MET with all the signals
        sample = 'MC_'+str(m)+'_'+str(ct)
        h_MET.fill(sample=sample, met=tree[sample].met)

ax = hist.plot1d(h_MET,density=True)
ax.set_ylabel("Fraction of events / bin width")
hep.cms.label(ax=ax,label="Preliminary")


# Q: What is the signal efficiencies for MET>200 GeV?

 - Use the following function `integral` the dense axes from `thres_lo` to infinity
 - Use the function `integral_low` the dense axes from lower bound to thres_high (for later parts)

**Answer (signal efficiency for MET > 200 GeV):** Run the cell below. The efficiency is the fraction of signal MC events with MET above 200 GeV. The lesson's checklist quotes a typical value around 1%, which is consistent with a generic MET trigger on a Higgs-portal signal. Report the value printed for each ctau sample; I did not execute the notebook, so the printed numbers are the reference.


In [ ]:
#  Integral the dense axes from thres_lo to infinity
def integral(h,sample,thres_lo):
    
    name = h.dense_axes()[0].name
    
    numer = h.integrate("sample",sample).integrate(name,slice(thres_lo,None),overflow="over").values()[()]
    denom = h.integrate("sample",sample).sum(name,overflow="over").values()[()]    
    
    return numer/denom

# Integral the dense axes from lower bound to thres_high
def integral_low(h,sample,thres_high):
    
    name = h.dense_axes()[0].name

    numer = h.integrate("sample",sample).integrate(name,slice(None,thres_high)).values()[()]
    denom = h.integrate("sample",sample).sum(name,overflow="over").values()[()]    

    return numer/denom

In [ ]:
# Choose from the following bin edges
h_MET.axis("met").edges()

In [ ]:
for m in mass:
    for ct in OLD_CTAU:
        sample = 'MC_'+str(m)+'_'+str(ct)
        # Fill the code
        eff = integral(h_MET, sample, 200)
        print("sample = %s, eff(MET>200) = %.5f "%(sample, eff))


# Event selections

The event selections for this search are kept at minimal to be as model independent as possible. 

The full event-level selections are 
 - MET trigger 
 - offline MET > 200 GeV
 - MET filters (make sure MET is not badly reconstructed)

For the ISR jet, the selections are
 - pT > 30 GeV
 - abs($\eta$) < 2.4
 - Pass tight jet Id

In [ ]:
def select_events(events,isData=False):
    
    ########### SELECTION: CLUSTERS ############
    ### At least 1 cluster (decay in CSC and matched to LLP for signal)
    ###
    if isData:
        sel_clusters = (events.cscRechitClusterSize)>0
    else:
        sel_clusters = (events.cscRechitCluster_match_gLLP==True) & (events.cscRechitCluster_match_gLLP_csc==True)

    ########### SELECTION: JETS ############
    sel_jet = (events.jetPt > 30) &\
              (abs(events.jetEta)<2.4) &\
              (events.jetTightPassId==True)
    
    ########### SELECTION: events ############
    # at least 1 selected cluster and jets
    # and pass MET selections
    sel_ev =  (events.METTrigger==True) &\
              (events.met>200) &\
              (events.metFilters==True) &\
              (ak.sum(sel_jet,axis=1)>=1) &\
              (ak.sum(sel_clusters,axis=1)>0)   
    
    return sel_clusters, sel_jet, sel_ev

# Cluster level selections

Aside from the MDS shower, there are not many handles in the events to reject background events.

Afterall, we only expect the shower, MET and an ISR jet in the signal events. 

Therefore, we will focus on using the MDS object features to enhance the signal to background ratio.  

First, we need to remove objects from the SM that could also create showers in the muon system. 

 - Punch-through jets
 - high pT muons


The dominant background from the main collision comes from punch-through jets that are not fully contained in the calorimeters and high pT muons that could create bremsstrahlung showers in the muon detectors. 

To remove those background, we reject clusters by matching them to reconstructed jets and muon. 

The pT of the jet/muon that the cluster is matched to are saved in objects called `cscRechitClusterJetVetoPt` 
and `cscRechitClusterMuonVetoPt`. 

In this exercise you will plot the two variables for signal and background and apply a selection on the two variables to remove clusters from jets and muons.

# Ex1 - plot jet veto and muon veto

 - Access the jet veto pT with `cscRechitClusterJetVetoPt`
 - Access the muon veto pT with `cscRechitClusterMuonVeto`

In [ ]:
nBins    = 50
bin_lo   = 0
bin_high = 500

h = hist.Hist("Events",
              hist.Cat("sample","sample"),
              hist.Bin("pt", "Jet veto pT[GeV]", nBins, bin_lo, bin_high)
             )

for i, sample in enumerate(tree.keys()):

    #See if we are treating data
    isData = True if ("data" in sample) else False

    # build the selection arrays
    sel_clusters, sel_jet, sel_ev = select_events(tree[sample],isData)

    # get the Jet veto Pt
    VetoPt = tree[sample].cscRechitClusterJetVetoPt

    # Event and cluster selections (make sure we have a cluster [and matched to LLP in CSC] )
    # Cluster selection first (jagged), then the event selection on the outer axis
    VetoPt = VetoPt[sel_clusters][sel_ev]

    # Select the VetoPt of the largest cluster in the event
    # fill_none puts -1 for events without clusters that satisfy the cluster cuts
    VetoPt = ak.fill_none(ak.firsts(VetoPt,axis=1),-1 )

    h.fill(sample=sample,pt=VetoPt)


In [ ]:
ax = hist.plot1d(h,density=True)
ax.set_ylabel("Fraction of events / bin width")
ax.set_yscale("log")
ax.set_ylim(1e-3,0.1)
hep.cms.label(ax=ax,label="Preliminary") 

# Q: What are the efficiencies Jet veto pT < 10 GeV?

**Answer (jet veto < 10 GeV efficiency):** Signal clusters are mostly not matched to a reconstructed jet, so the signal efficiency should be close to 1. The data (background) fraction passing is the fraction of clusters with no nearby high-pT jet, so the cut removes most punch-through jet clusters while keeping most signal. Quote the printed numbers for your run. Note that events without a selected cluster are filled at -1 and fall in the underflow, so they are counted in the numerator of `integral_low`.


In [ ]:
# Hint: use integral_low and integral

for i, sample in enumerate(tree.keys()):
    # Fill code here
    eff = integral_low(h, sample, 10)
    print("sample = %s, eff(VetoPT<10) = %.5f "%(sample, eff))


# Plot muon veto 

In [ ]:
# Think about the bin ranges and number of bins you want to in the histogram before plotting
# 50 bins of 2 GeV: the 20 GeV threshold falls on a bin edge
nBins    = 50
bin_lo   = 0
bin_high = 100

h = hist.Hist("Events",
              hist.Cat("sample","sample"),
              hist.Bin("pt", "Muon veto pT[GeV]", nBins, bin_lo, bin_high)
             )


for i, sample in enumerate(tree.keys()):

    #See if we are treating data
    isData = True if ("data" in sample) else False

    # build the selection arrays
    sel_clusters, sel_jet, sel_ev = select_events(tree[sample],isData)

    # get the Muon veto Pt
    VetoPt = tree[sample].cscRechitClusterMuonVetoPt

    # Event and cluster selections (make sure we have a cluster [and matched to LLP in CSC] )
    VetoPt = VetoPt[sel_clusters][sel_ev]

    # Select the VetoPt of the largest cluster
    # fill_none puts -1 for events without clusters that satisfy the cluster cuts
    VetoPt = ak.fill_none(ak.firsts(VetoPt,axis=1),-1 )

    h.fill(sample=sample,pt=VetoPt)


In [ ]:
ax = hist.plot1d(h,density=True)
ax.set_ylabel("Fraction of events / bin width")
ax.set_yscale("log")
ax.set_ylim(1e-5,0.1)
hep.cms.label(ax=ax,label="Preliminary") 

# Q: What are the efficiencies Muon veto pT < 20 GeV?

**Answer (muon veto < 20 GeV efficiency):** Similar to the jet veto: signal clusters are rarely matched to a muon with pT > 20 GeV, so the signal efficiency is high, while the data distribution is suppressed by the cut. Quote the printed numbers from your run.


In [ ]:
# Hint: use integral_low and integral

for i, sample in enumerate(tree.keys()):
    # Fill code here
    eff = integral_low(h, sample, 20)
    print("sample = %s, eff(Muon VetoPT < 20) = %.5f "%(sample, eff))


# Ex2 - plot number of ME11 and ME12 hits in the cluster

Additionally, punch-through jets or muon bremsstrahlung showers *might not get reconstructed as jets and muons*.

To fully remove these background, we remove clusters that have hits in the first CSC stations (ME11/ME12) that have little shielding in front.

In this exercise, you will plot the number of ME11/ME12 hits in clusters for signal and background and apply a selection on the two variables to remove clusters from jets and muons.

## plot number of ME11 and ME12 hits in cluster

 - Access the number of chamber hits in the cluster with the following variables:
   - `cscRechitClusterNRechitChamberPlus11`
   - `cscRechitClusterNRechitChamberMinus11`
   - `cscRechitClusterNRechitChamberPlus12`
   - `cscRechitClusterNRechitChamberMinus12`

In [ ]:
# Integer bins 0..10 so that the N=0 bin is [0,1)
nBins    = 10
bin_lo   = 0
bin_high = 10

h = hist.Hist("Events",
              hist.Cat("sample","sample"),
              hist.Bin("nME11_12", r"$N_{Me11/12 hits}$", nBins, bin_lo, bin_high)
             )


for i, sample in enumerate(tree.keys()):

    #See if we are treating data
    isData = True if ("data" in sample) else False

    # build the selection arrays
    sel_clusters, sel_jet, sel_ev = select_events(tree[sample],isData)

    # Compute nME11_12 (sum over the four ME11/ME12 chamber hit counts)
    nME11_12 = (tree[sample].cscRechitClusterNRechitChamberPlus11
              + tree[sample].cscRechitClusterNRechitChamberMinus11
              + tree[sample].cscRechitClusterNRechitChamberPlus12
              + tree[sample].cscRechitClusterNRechitChamberMinus12)

    # Event and cluster selections (make sure we have a cluster [and matched to LLP in CSC] )
    nME11_12 = nME11_12[sel_clusters][sel_ev]

    # Select the VetoPt of the largest cluster
    # fill_none puts -1 for events without clusters that satisfy the cluster cuts
    nME11_12 = ak.fill_none(ak.firsts(nME11_12,axis=1),-1 )

    h.fill(sample=sample,nME11_12=nME11_12)


In [ ]:
ax = hist.plot1d(h,density=True)
ax.set_ylabel("Fraction of events / bin width")
ax.set_yscale("log")
ax.set_ylim(1e-4,1)
hep.cms.label(ax=ax,label="Preliminary") 

# Q: What are the efficiencies for nME11_12 ==0?

**Answer (nME11_12 == 0 efficiency):** Signal clusters from an LLP decaying in the outer muon system rarely produce hits in ME11/ME12, which are shielded by little material in front. Punch-through jets and brems showers do. The signal efficiency should be high and the data fraction small. Quote the printed numbers. `integral_low(h, sample, 1)` integrates the [0,1) bin, which is the N = 0 bin; events without a cluster are in the underflow, which inflates the efficiencies slightly.


In [ ]:
# Hint: use integral_low and integral

for i, sample in enumerate(tree.keys()):
    # Fill code here
    eff = integral_low(h, sample, 1)
    print("sample = %s, eff(nME11_12 ==0) = %.5f "%(sample, eff))


# Ex3 - plot cluster eta

After we removed punch-through jets and muon brems, we observed that there are a lot more backgorund in higer $\eta$ region, where the muon reconstruction efficiency is lower and more pileup particles are present to create clusters.

In this exercise you will plot the cluster $\eta$ for signal and background and apply a selection on the variable to remove clusters from high $\eta$ region.

In [ ]:
# 60 bins of width 0.1: the |eta| = 2 edges fall on bin edges
nBins    = 60
bin_lo   = -3
bin_high = 3

h = hist.Hist("Events",
              hist.Cat("sample","sample"),
              hist.Bin("eta", r"cluster $\eta$", nBins, bin_lo, bin_high)
             )


for i, sample in enumerate(tree.keys()):

    #See if we are treating data
    isData = True if ("data" in sample) else False

    # build the selection arrays
    sel_clusters, sel_jet, sel_ev = select_events(tree[sample],isData)

    # Get cluster Eta
    clsEta = tree[sample].cscRechitClusterEta

    # Event and cluster selections (make sure we have a cluster [and matched to LLP in CSC] )
    clsEta = clsEta[sel_clusters][sel_ev]

    # Select the eta of the largest cluster
    # fill_none puts -999 (not -1, which would fall inside |eta|<2) for events without clusters
    clsEta = ak.fill_none(ak.firsts(clsEta,axis=1),-999 )

    h.fill(sample=sample,eta=clsEta)


In [ ]:
ax = hist.plot1d(h,density=True)
ax.set_ylabel("Fraction of events / bin width")
ax.set_yscale("log")
ax.set_ylim(1e-4,10)
hep.cms.label(ax=ax,label="Preliminary") 

# Q: What are the efficiencies for $abs(\eta)<2$ ?

**Answer (|eta| < 2 efficiency):** The efficiency is the fraction of clusters with |eta| < 2 (computed as the difference of two `integral_low` calls). Signal clusters are concentrated at central eta, so the efficiency is high, and the data fraction is reduced by the cut. Quote the printed numbers.


In [ ]:
# Hint: use integral_low and integral
# |eta|<2 = (fraction below +2) - (fraction below -2); the underflow (-999) cancels in the difference

for i, sample in enumerate(tree.keys()):
    # Fill code here
    eff = integral_low(h, sample, 2) - integral_low(h, sample, -2)
    print(r'sample = %s, eff(abs($\eta$)<2) = %.5f '%(sample, eff))


# Ex4 - plot cluster time (after vetos are applied)

The remaining background clusters after punch-through jet and muon brem showers from the main collision are removed, are from low pT particles in pileup events.

To verify this, you will plot the cluster time for signal and background, after applying the vetoes, in this exercise to check for any out-of-time pileup contributions in data.


In [ ]:
nBins    = 100
bin_lo   = -25
bin_high = 25

h = hist.Hist("Events",
              hist.Cat("sample","sample"),
              hist.Bin("time", r"cluster time[ns]", nBins, bin_lo, bin_high)
             )


for i, sample in enumerate(tree.keys()):

    #See if we are treating data
    isData = True if ("data" in sample) else False

    # build the selection arrays
    sel_clusters, sel_jet, sel_ev = select_events(tree[sample],isData)

    # Get Variables
    clsTime  = tree[sample].cscRechitClusterTime
    jetVeto  = tree[sample].cscRechitClusterJetVetoPt
    muonVeto = tree[sample].cscRechitClusterMuonVetoPt
    clsEta   = tree[sample].cscRechitClusterEta

    # Apply the following selections:
    # - cluster and event selections
    # - JetVeto < 10 GeV
    # - MuonVeto < 20 GeV
    # - cluster eta < 2
    # Try running without Jet and muon Veto as well! (drop the jetVeto / muonVeto terms)
    sel_cls = sel_clusters & (jetVeto < 10) & (muonVeto < 20) & (np.abs(clsEta) < 2)
    clsTime = clsTime[sel_cls][sel_ev]

    # Select the time of the largest cluster
    # fill_none puts -999 for events without clusters that satisfy the cluster cuts
    clsTime = ak.fill_none(ak.firsts(clsTime,axis=1),-999 )

    h.fill(sample=sample,time=clsTime)


In [ ]:
ax = hist.plot1d(h,density=True)
ax.set_ylabel("Fraction of events / bin width")
hep.cms.label(ax=ax,label="Preliminary") 

# Q: What are the efficiencies for $-5<t<12.5$ ns ?

**Answer (-5 < t < 12.5 ns efficiency):** The signal peaks in-time (near t = 0 within the window), so the efficiency is high for signal. The data fraction in-window is smaller because a sizeable part of the background lives in the out-of-time (early or late) bins. Quote the printed numbers.


In [ ]:
def integral_range(h,sample,thres_lo,thres_high):

    name = h.dense_axes()[0].name

    numer = h.integrate("sample",sample).integrate(name,slice(thres_lo,thres_high)).values()[()]
    denom = h.integrate("sample",sample).sum(name,overflow="over").values()[()]

    return numer/denom

for i, sample in enumerate(tree.keys()):
    # Fill code here
    eff = integral_range(h, sample, -5, 12.5)
    print(r'sample = %s, eff( -5 < t < 12.5 ns) = %.5f '%(sample, eff))


# Q: Can you suggest an out-of-time validation region based on the above plot?

A good validation region needs to have:
 - No (or at least small) signal conteminations (i.e. signal depleted) 
 - Similar background composition as your signal-rich region

**Answer (out-of-time validation region):** Based on the cluster-time plot after the vetoes, the early out-of-time region t < -12.5 ns is a good candidate. It is signal depleted, since signal clusters are in-time (the signal window is -5 < t < 12.5 ns), and it is populated by pileup clusters, so its background composition is similar to the signal region. This is the region implemented in Ex5 for data.


# Ex5 - plot $\Delta\phi\mathrm{(cluster, MET)}$ and $N_{hits}$ in Out-of-time Validation Region (final discriminating variables)


The final discriminating variables that we will use to extract the signal and estimate background are 
 - the number of hits in the cluster ($N_{\text{hits}}$) and
 - the azimuthal angle between the cluster and MET ( $\Delta\phi\text{(cluster, MET)}$).

The background estimation method will be described in more detail in the next episode.

In this exercise, we will just plot the distributions of the two variables, to understand the shape of the two variables.

## Plot data only in OOT region to avoid unblinding!

## Implement the OOT selection you came up with above!

## plot $N_{hits}$

In [ ]:
nBins    = 30
bin_lo   = 0
bin_high = 300

h = hist.Hist("Events",
              hist.Cat("sample","sample"),
              hist.Bin("nhits", r"$N_{hits}$", nBins, bin_lo, bin_high)
             )


for i, sample in enumerate(tree.keys()):

    #See if we are treating data
    isData = True if ("data" in sample) else False

    # build the selection arrays
    sel_clusters, sel_jet, sel_ev = select_events(tree[sample],isData)

    # Get Variables
    nhits    = tree[sample].cscRechitClusterSize
    clsTime  = tree[sample].cscRechitClusterTime
    jetVeto  = tree[sample].cscRechitClusterJetVetoPt
    muonVeto = tree[sample].cscRechitClusterMuonVetoPt
    clsEta   = tree[sample].cscRechitClusterEta

    # Only plot Out-of-Time(OOT) region for data (t < -12.5 ns, early OOT)
    # For MC (signal) use the in-time signal window -5 < t < 12.5 ns
    if isData:
        sel_time = clsTime < -12.5
    else:
        sel_time = (clsTime > -5) & (clsTime < 12.5)

    # Apply the following selections:
    # - cluster and event selections
    # - JetVeto < 10 GeV
    # - MuonVeto < 20 GeV
    # - cluster eta < 2
    # - OOT selection for data; in-time selection for MC
    sel_cls = sel_clusters & (jetVeto < 10) & (muonVeto < 20) & (np.abs(clsEta) < 2) & sel_time
    nhits = nhits[sel_cls][sel_ev]

    # Select the value of the largest cluster
    # fill_none puts -1 for events without clusters that satisfy the cluster cuts
    nhits = ak.fill_none(ak.firsts(nhits,axis=1),-1 )

    h.fill(sample=sample,nhits=nhits)


In [ ]:
ax = hist.plot1d(h,density=True)
ax.set_ylabel("Fraction of events / bin width")
ax.set_yscale("log")
ax.set_ylim(1e-4,0.1)
hep.cms.label(ax=ax,label="Preliminary") 

# Q: What are the efficiencies for $N_{hits}>130$ ?


**Answer (N_hits > 130 efficiency):** Signal MDS showers have many hits, so the signal efficiency for N_hits > 130 is moderate-to-high, while the pileup background (small clusters) is strongly suppressed. Use the printed numbers for each sample.


In [ ]:
for i, sample in enumerate(tree.keys()):
    # Fill code here
    eff = integral(h, sample, 130)
    print(r'sample = %s, eff($N_{hits}>130$) = %.5f '%(sample, eff))


## plot abs($\Delta\phi\mathrm{(cluster, MET)}$)

In [ ]:
# 64 bins of width 0.05 on [0, 3.2]: the 0.75 threshold falls on a bin edge
nBins    = 64
bin_lo   = 0
bin_high = 3.2

h = hist.Hist("Events",
              hist.Cat("sample","sample"),
              hist.Bin("dphi", r"$\Delta\phi(cls,MET)$", nBins, bin_lo, bin_high)
             )


for i, sample in enumerate(tree.keys()):

    #See if we are treating data
    isData = True if ("data" in sample) else False

    # build the selection arrays
    sel_clusters, sel_jet, sel_ev = select_events(tree[sample],isData)

    # Get Variables
    dphi     = tree[sample].cscRechitClusterMet_dPhi
    clsTime  = tree[sample].cscRechitClusterTime
    jetVeto  = tree[sample].cscRechitClusterJetVetoPt
    muonVeto = tree[sample].cscRechitClusterMuonVetoPt
    clsEta   = tree[sample].cscRechitClusterEta

    # Only plot Out-of-Time(OOT) region for data, in-time window for MC
    if isData:
        sel_time = clsTime < -12.5
    else:
        sel_time = (clsTime > -5) & (clsTime < 12.5)

    # Apply the selections (cluster, event, vetoes, |eta|<2, OOT/in-time)
    sel_cls = sel_clusters & (jetVeto < 10) & (muonVeto < 20) & (np.abs(clsEta) < 2) & sel_time

    dphi = np.abs(dphi[sel_cls][sel_ev])

    # Select the value of the largest cluster
    # fill_none puts -999 for events without clusters that satisfy the cluster cuts
    dphi = ak.fill_none(ak.firsts(dphi,axis=1),-999 )

    h.fill(sample=sample,dphi=dphi)


In [ ]:
ax = hist.plot1d(h,density=True)
ax.set_ylabel("Fraction of events / bin width")
hep.cms.label(ax=ax,label="Preliminary") 

# Q: What are the efficiencies for $abs(\Delta\phi(cls,MET)<0.75)$ ?


**Answer (|dPhi(cls, MET)| < 0.75 efficiency):** Signal clusters are back-to-back with the Higgs, so the azimuthal angle between the cluster and MET peaks at 0 and the efficiency for |dPhi| < 0.75 is high. Background is flat in dPhi, so the cut rejects a large fraction of it. Quote the printed numbers.


In [ ]:
for i, sample in enumerate(tree.keys()):
    # Fill code here
    eff = integral_low(h, sample, 0.75)
    print(r'sample = %s, eff($abs(\Delta\phi(cls,MET)<0.75)$) = %.5f '%(sample, eff))
